# Energy-Efficient Beamforming for ISAC Systems

This notebook demonstrates the algorithms from:

> **Jiaqi Zou, Songlin Sun, Christos Masouros, Yuanhao Cui**, "Energy-Efficient Beamforming Design for Integrated Sensing and Communications Systems," *IEEE Transactions on Communications*, 2024.

**arXiv**: https://arxiv.org/abs/2307.04002

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.system_model import ISACSystemModel
from src.dinkelbach_solver import DinkelbachSolver
from src.pareto_optimizer import ParetoOptimizer
from src.baselines import run_all_baselines
from src.ee_metrics import compute_ee_c, compute_ee_s, compute_crb

## 1. System Model Setup

In [ ]:
# System parameters (Section VI of the paper)
model = ISACSystemModel(
    M=16,              # Transmit antennas
    K=4,               # Communication users
    N=20,              # Receive antennas for sensing
    P_max_dbm=30,      # Max power: 30 dBm = 1 Watt
    P0_dbm=33,         # Circuit power: 33 dBm ≈ 2 Watts
    epsilon=0.35,      # PA efficiency
    L=30,              # Frame length
    seed=42
)

print(f"System: M={model.M}, K={model.K}, N={model.N}")
print(f"Max Power: {model.P_max:.4f} W ({10*np.log10(model.P_max*1000):.1f} dBm)")
print(f"Circuit Power: {model.P0:.4f} W ({10*np.log10(model.P0*1000):.1f} dBm)")

## 2. Communication EE Maximization (Algorithm 1)

In [ ]:
# Run Dinkelbach solver
solver = DinkelbachSolver(model, max_dinkelbach_iter=20, verbose=True)
result = solver.solve(target_angle_deg=90.0)

print(f"\nResults:")
print(f"  Converged: {result.converged}")
print(f"  Iterations: {result.n_iterations}")
print(f"  EE_C: {result.ee_c:.4f} bits/Hz/J")
print(f"  Sum Rate: {result.sum_rate:.4f} bits/Hz")
print(f"  Total Power: {result.total_power:.4f} W")

### Convergence Plot

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(result.obj_history)+1), result.obj_history, 'b-o')
plt.xlabel('Dinkelbach Iteration')
plt.ylabel('$EE_C$ (bits/Hz/J)')
plt.title('Convergence of Communication EE')
plt.grid(True, alpha=0.3)
plt.show()

## 3. Baseline Comparison

In [ ]:
# Run all baseline schemes
baseline_results = run_all_baselines(model, target_angle_deg=90.0)

print(f"{'Scheme':<25} {'EE_C':>12} {'EE_S':>12} {'Sum Rate':>12}")
print("-" * 65)

for name, res in baseline_results.items():
    print(f"{name:<25} {res.ee_c:>12.4f} {res.ee_s:>12.4f} {res.sum_rate:>12.4f}")

print(f"\n{'Proposed (Dinkelbach)':<25} {result.ee_c:>12.4f}")

## 4. Pareto Boundary (Algorithm 4)

In [ ]:
# Trace Pareto boundary
optimizer = ParetoOptimizer(model, n_pareto_points=15, verbose=True)
pareto_points = optimizer.trace_pareto_boundary(target_angle_deg=90.0)

print(f"\nFound {len(pareto_points)} Pareto-optimal points")

In [ ]:
# Plot Pareto boundary
plt.figure(figsize=(8, 6))

ee_c_vals = [p.ee_c for p in pareto_points]
ee_s_vals = [p.ee_s for p in pareto_points]

plt.plot(ee_c_vals, ee_s_vals, 'b-o', markersize=8, linewidth=2, label='Pareto Boundary')
plt.plot(ee_c_vals[0], ee_s_vals[0], 'gs', markersize=12, label='EE$_S$ Max')
plt.plot(ee_c_vals[-1], ee_s_vals[-1], 'r^', markersize=12, label='EE$_C$ Max')

plt.xlabel('$EE_C$ (bits/Hz/J)')
plt.ylabel('$EE_S$ (1/J)')
plt.title('Pareto Boundary: Communication vs Sensing EE')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Beamforming Pattern Visualization

In [ ]:
# Compute beampattern
angles_deg = np.linspace(0, 180, 181)
angles_rad = np.radians(angles_deg)

W_opt = result.W
beampattern = []

for theta in angles_rad:
    a_t = model.steering_vector_tx(theta)
    # Total radiated power at angle theta
    power = np.abs(a_t.conj() @ W_opt @ W_opt.conj().T @ a_t)
    beampattern.append(power)

beampattern = np.array(beampattern)
beampattern_db = 10 * np.log10(beampattern / np.max(beampattern) + 1e-15)

plt.figure(figsize=(10, 5))
plt.plot(angles_deg, beampattern_db, 'b-', linewidth=2)
plt.axvline(x=90, color='r', linestyle='--', label='Target (90°)')
plt.xlabel('Angle (degrees)')
plt.ylabel('Normalized Power (dB)')
plt.title('Transmit Beampattern')
plt.ylim([-30, 5])
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. Parameter Sensitivity Analysis

In [ ]:
# Vary number of transmit antennas
M_values = [8, 12, 16, 20]
ee_c_values = []

for M in M_values:
    model_m = ISACSystemModel(M=M, K=4, N=20, seed=42)
    solver_m = DinkelbachSolver(model_m, max_dinkelbach_iter=10, verbose=False)
    result_m = solver_m.solve(target_angle_deg=90.0)
    ee_c_values.append(result_m.ee_c)

plt.figure(figsize=(8, 5))
plt.plot(M_values, ee_c_values, 'b-o', markersize=8, linewidth=2)
plt.xlabel('Number of Transmit Antennas (M)')
plt.ylabel('$EE_C$ (bits/Hz/J)')
plt.title('EE$_C$ vs Number of Antennas')
plt.grid(True, alpha=0.3)
plt.show()

## Summary

This notebook demonstrated:

1. **Algorithm 1 (Dinkelbach)**: Communication EE maximization with convergence in ~10 iterations
2. **Baseline Comparison**: Proposed method outperforms baselines
3. **Algorithm 4 (Pareto)**: Tradeoff between communication and sensing EE
4. **Beampattern**: Shows energy-efficient beamforming pattern
5. **Sensitivity**: EE_C improves with more antennas